# 01 - Veri Keşfi

Bu not defteri, veri setini tanımak için ilk adımı içerir. Amaç; verinin genel yapısını görmek, sınıf dağılımını anlamak ve modelleme öncesi dikkat edilmesi gereken noktaları netleştirmektir.

Bu bölüm sonunda şu sorulara cevap alacağız:
- Veri dengeli mi?
- Hangi alanlarda sıfır değer problemi var?
- Hedef değişken ile ilişkisi görece güçlü özellikler hangileri?


In [ ]:
# Ortamı hazırlıyor ve proje kökünü Python yoluna ekliyoruz.
from pathlib import Path
import sys

proje_kok = Path.cwd().resolve()
while proje_kok.name != 'diyabet_risk_tahmini' and proje_kok.parent != proje_kok:
    proje_kok = proje_kok.parent

if proje_kok.name != 'diyabet_risk_tahmini':
    raise RuntimeError('Notebook proje kökünden veya alt klasörlerinden çalıştırılmalıdır.')

if str(proje_kok) not in sys.path:
    sys.path.insert(0, str(proje_kok))

print(f'Proje kökü: {proje_kok}')


In [ ]:
# Analiz boyunca kullanacağımız kütüphaneleri ve proje içi yardımcı fonksiyonları içe aktarıyoruz.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from makine_ogrenmesi.kaynak.veri_yukleyici import veri_setini_yukle
from makine_ogrenmesi.kaynak.ozellik_yapilandirmasi import (
    HEDEF_KOLONU,
    OZELLIK_KOLONLARI,
    SIFIRI_EKSIK_SAYILAN_KOLONLAR,
)

plt.style.use('default')


In [ ]:
# Veri dosyasını yüklüyor, boyutunu kontrol ediyor ve ilk satırları görüntülüyoruz.
veri_yolu = proje_kok / 'makine_ogrenmesi' / 'veri' / 'ham' / 'diabetes.csv'
veri = veri_setini_yukle(veri_yolu)

print('Veri boyutu:', veri.shape)
print('Kolonlar:', list(veri.columns))
veri.head()


In [ ]:
# Kolon tipleri, boş değer sayıları ve benzersiz değer adetleriyle temel bir kalite özeti çıkarıyoruz.
genel_ozet = pd.DataFrame({
    'veri_tipi': veri.dtypes.astype(str),
    'boş_değer_sayısı': veri.isna().sum(),
    'benzersiz_değer_sayısı': veri.nunique(),
})
genel_ozet


In [ ]:
# Hedef sınıf dağılımını adet ve oran olarak hesaplıyoruz.
sinif_sayim = veri[HEDEF_KOLONU].value_counts().sort_index()
sinif_oran = veri[HEDEF_KOLONU].value_counts(normalize=True).sort_index()

sinif_tablosu = pd.DataFrame({'adet': sinif_sayim, 'oran': sinif_oran})
sinif_tablosu.index = ['sınıf_0', 'sınıf_1']
sinif_tablosu


In [ ]:
# Sınıf dağılımını görsel olarak inceleyerek dengesizlik olup olmadığını hızlıca yorumluyoruz.
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(['Sınıf 0', 'Sınıf 1'], sinif_sayim.values, color=['#4C78A8', '#E45756'])
ax.set_title('Hedef Sınıf Dağılımı')
ax.set_ylabel('Örnek Sayısı')
ax.grid(axis='y', alpha=0.3)
plt.show()


In [ ]:
# Sıfır değeri eksik kabul ettiğimiz kolonlarda problem büyüklüğünü ölçüyoruz.
sifir_sayilari = (veri[SIFIRI_EKSIK_SAYILAN_KOLONLAR] == 0).sum().sort_values(ascending=False)
sifir_oranlari = (sifir_sayilari / len(veri)).rename('oran')

sifir_ozeti = pd.concat([sifir_sayilari.rename('adet'), sifir_oranlari], axis=1)
sifir_ozeti


In [ ]:
# Sıfır değer oranlarını grafikte göstererek hangi alanların daha kritik olduğunu vurguluyoruz.
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(sifir_ozeti.index, sifir_ozeti['oran'], color='#72B7B2')
ax.set_title('Sıfır Değer Oranları (Eksik Kabul Edilen Kolonlar)')
ax.set_ylabel('Oran')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=30)
plt.show()


In [ ]:
# Her özelliğin dağılımını histogramlarla birlikte inceliyoruz.
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.ravel()
for idx, kolon in enumerate(OZELLIK_KOLONLARI):
    axes[idx].hist(veri[kolon], bins=20, color='#4C78A8', alpha=0.8)
    axes[idx].set_title(kolon)
    axes[idx].grid(alpha=0.2)

plt.suptitle('Özellik Dağılımları', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Korelasyon matrisini hesaplayıp ısı haritası üzerinden genel ilişki yapısını görüyoruz.
korelasyon = veri[OZELLIK_KOLONLARI + [HEDEF_KOLONU]].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(korelasyon, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(korelasyon.columns)))
ax.set_xticklabels(korelasyon.columns, rotation=90)
ax.set_yticks(range(len(korelasyon.index)))
ax.set_yticklabels(korelasyon.index)
ax.set_title('Korelasyon Matrisi')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


In [ ]:
# Outcome ile en güçlü ve en zayıf ilişkileri sıralayıp kısa bir öncelik listesi oluşturuyoruz.
outcome_korelasyon = korelasyon[HEDEF_KOLONU].drop(HEDEF_KOLONU).sort_values(ascending=False)
print('Outcome ile en yüksek pozitif ilişki:')
print(outcome_korelasyon.head(3))

print('\nOutcome ile en düşük ilişki:')
print(outcome_korelasyon.tail(3))


## Kısa değerlendirme

- Veri seti görece küçük olduğu için metriklerin oynak olması doğaldır.
- Sıfır değer problemi on işleme kalitesini doğrudan etkiler.
- Bu nedenle bir sonraki adımda 0→NaN, imputasyon ve ölçekleme adımlarını dikkatli doğrulayacağız.
